# 🚀 LineVul Multi-Split Training on Google Colab (Free T4 GPU)

This notebook trains your fine-tuned **LineVul** vulnerability detection model on your new datasets (`kernel data` and `dataset new`).

### 📁 The 3 Generated Dataset Splits:
1. **Split 1 (`split_1_kernel`)**: Linux Kernel vulnerabilities (`vulnerable_function` vs. `nonvulnerable_function`). Specializes in low-level drivers, memory corruptions, race conditions, buffer overflows, and pointer arithmetic.
2. **Split 2 (`split_2_application`)**: User-space and open-source applications (`dataset new`). Specializes in general C/C++ libraries, parsing errors, integer overflows, format strings, and network protocols.
3. **Split 3 (`split_3_combined`)**: The complete unified dataset combining both Kernel + Application vulnerabilities into a single master benchmark.

---

### 📌 Prerequisites Before Running:
1. Hardware Accelerator: Go to **Runtime → Change runtime type → T4 GPU** (under Hardware accelerator, 100% Free).
2. Upload your prepared `linevul_train.zip` to Google Drive or directly to the Colab files panel.

## 1. Verify Free GPU Acceleration
Verify that a Tesla T4 GPU (15 GB VRAM) is assigned.

In [ ]:
!nvidia-smi
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
else:
    raise SystemExit('⚠️ GPU not enabled! Go to Runtime -> Change runtime type -> Select T4 GPU.')

## 2. Mount Google Drive
Mount Google Drive so all checkpoints and logs are automatically saved directly to your Drive, preventing any lost work.

In [ ]:
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive')
DRIVE_DIR = Path('/content/drive/MyDrive/linevul_training')
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
print('Drive mounted successfully. Checkpoints will be saved to:', DRIVE_DIR)

## 3. Extract Project & Dataset Splits
Unzips `linevul_train.zip` into `/content/`.

In [ ]:
import glob, os

# Locate linevul_train.zip in Drive or /content
candidates = glob.glob('/content/drive/MyDrive/**/linevul_train.zip', recursive=True) + \
             glob.glob('/content/drive/MyDrive/**/linevul.zip', recursive=True) + \
             glob.glob('/content/linevul_train.zip')

if candidates:
    zip_path = candidates[0]
    print(f'Found project archive: {zip_path}')
    !unzip -qo "{zip_path}" -d /content/
else:
    raise FileNotFoundError('⚠️ linevul_train.zip not found! Please upload it to your Google Drive.')

# Set working directory
for d in ['/content/linevul/linevul', '/content/linevul', '/content']:
    if os.path.exists(os.path.join(d, 'linevul', 'train.py')):
        os.chdir(d)
        print(f'Working directory set to: {os.getcwd()}')
        break

# Install dependencies
!pip install -q transformers scikit-learn tqdm

## 4. Inspect Available Splits
Shows sample counts and validation pairs for each split.

In [ ]:
import json
from pathlib import Path

summary_file = Path('data/splits_summary.json')
if summary_file.exists():
    summary = json.loads(summary_file.read_text())
    print(json.dumps(summary, indent=2))
else:
    print('Available folders in data/:', [p.name for p in Path('data').iterdir() if p.is_dir()])

---
## 🏋️‍♂️ Step-by-Step Training for Each Split

You can choose to train **any individual split**, or train them **progressively**.

### Option A: Train Split 1 (Kernel Vulnerabilities Only)
- **Focus**: Linux Kernel drivers, networking stack, buffer checks, and kernel memory.
- **Runtime on Colab T4**: ~35–50 minutes.

In [ ]:
OUT_SPLIT1 = '/content/drive/MyDrive/linevul_training/checkpoint_split1_kernel'

!python -m linevul.train \
  --data-dir data/split_1_kernel \
  --out-dir "{OUT_SPLIT1}" \
  --epochs 5 \
  --batch-size 8 \
  --lr 2e-5 \
  --stmtweight auto \
  --patience 3 \
  --ranking-lambda 0.5 \
  --ranking-margin 1.0

# Evaluate on Kernel Test Set
!python scripts/eval_gate.py --checkpoint "{OUT_SPLIT1}"

### Option B: Train Split 2 (Application / Open-Source Vulnerabilities Only)
- **Focus**: User-space libraries, string manipulation (strcpy, sprintf), integer overflows.
- **Runtime on Colab T4**: ~20–30 minutes.

In [ ]:
OUT_SPLIT2 = '/content/drive/MyDrive/linevul_training/checkpoint_split2_application'

!python -m linevul.train \
  --data-dir data/split_2_application \
  --out-dir "{OUT_SPLIT2}" \
  --epochs 5 \
  --batch-size 8 \
  --lr 2e-5 \
  --stmtweight auto \
  --patience 3 \
  --ranking-lambda 0.5 \
  --ranking-margin 1.0

# Evaluate on Application Test Set
!python scripts/eval_gate.py --checkpoint "{OUT_SPLIT2}"

### Option C: Train Split 3 (Full Combined Master Benchmark - Recommended)
- **Focus**: Comprehensive vulnerability detection covering both Kernel and Application domains.
- **Runtime on Colab T4**: ~1 to 1.5 hours.

In [ ]:
OUT_SPLIT3 = '/content/drive/MyDrive/linevul_training/checkpoint_split3_combined'

!python -m linevul.train \
  --data-dir data/split_3_combined \
  --out-dir "{OUT_SPLIT3}" \
  --epochs 5 \
  --batch-size 8 \
  --lr 2e-5 \
  --stmtweight auto \
  --patience 3 \
  --ranking-lambda 0.5 \
  --ranking-margin 1.0

# Evaluate on Combined Test Set
!python scripts/eval_gate.py --checkpoint "{OUT_SPLIT3}"

## 5. Download the Desired Checkpoint to Local Machine
Select which checkpoint to zip and download.

In [ ]:
import os
from google.colab import files

# Choose which model to download: OUT_SPLIT3 (recommended), OUT_SPLIT1, or OUT_SPLIT2
SELECTED_CHECKPOINT = OUT_SPLIT3

!zip -r /content/linevul_selected_checkpoint.zip "{SELECTED_CHECKPOINT}"
print('Download starting...')
files.download('/content/linevul_selected_checkpoint.zip')